# SLM — train + eval the transformation-error diagnoser (self-contained Colab)

**Upload just this notebook + your dataset zip, then Run all.** This notebook writes its own
scoring code, so there is no package to upload and no paths to edit.

- **Base model:** Qwen3-VL-4B (4-bit) — fits a free **T4**.
- **Defaults tuned to finish on a free T4 in ~1 hour:** 500 training steps + a fixed 300-record
  eval sample. Both are labeled below — scale them up for your final numbers.

**Steps:** Runtime → Change runtime type → **T4 GPU** → then **Runtime → Run all**.
When prompted, upload `transform_diagnosis_data.zip` in the Files pane (left) and re-run the
data cell.

In [ ]:
# Confirm a GPU is attached (expect a T4 on free Colab).
!nvidia-smi

In [ ]:
# Install Unsloth — AUTO-SKIPS if it's already available (e.g. you set up a conda env on an
# HPC like MIT ORCD). On a bare Colab runtime it installs it. Safe to Run All either way.
import importlib.util, subprocess, sys
if importlib.util.find_spec("unsloth") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir",
                    "unsloth", "unsloth_zoo"])
    print("installed unsloth — if torch was upgraded, Runtime > Restart session, then Run all again")
else:
    print("unsloth already available — skipping install")

In [ ]:
# --- Write the eval harness's code (no upload needed) --------------------------------------
# Creates a tiny `slm_eval` package on this session. The next three cells write the exact
# source of the scoring modules; this one just makes the folder.
import os
os.makedirs("slm_eval", exist_ok=True)
open("slm_eval/__init__.py", "w").close()
print("created slm_eval/ package")

In [ ]:
%%writefile slm_eval/transform_core.py
"""
transform_core — THE single canonical implementation of rigid-motion transforms,
grading, and student-error diagnosis for the composed-transformation dataset.

Everything else in the package imports from here. There is exactly ONE implementation
of transforms / grade / diagnose / recover_map in the whole package. If judging
behaviour must change, change THIS file and its contract test (`test_transform_core.py`)
only — never fork it.

Design
------
A transform is an affine map on the integer lattice::

    p -> M @ p + t

where ``M`` is an integer 2x2 orthogonal matrix ``((a, b), (c, d))`` (row-major) and
``t = (e, f)`` is an integer translation vector. All arithmetic is exact integer
arithmetic; no floats ever appear in the transform / grade / label path.

``det(M)`` is the orientation:

* ``+1`` — orientation preserving (rotation, including the identity)
* ``-1`` — orientation reversing (reflection)

Composition
-----------
``compose(seq)`` composes a sequence where ``seq[0]`` is applied FIRST. For an affine
map, applying ``T1`` then ``T2`` gives ``p -> M2 @ (M1 @ p + t1) + t2``, i.e.
``M = M2 @ M1`` and ``t = M2 @ t1 + t2``.

Text forms
----------
Every transform has a human-readable string form (see ``describe_transform``) matching
the dataset schema, e.g. ``"rotate 90 degrees counterclockwise"``, ``"translate 7 left"``,
``"reflect across x axis"``. ``parse_transform`` is the inverse. ``compose`` / ``grade`` /
``diagnose`` accept either ``Transform`` objects or these strings (strings are parsed),
so the dataset can be verified directly from its stored text.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Iterable, List, Optional, Sequence, Tuple, Union

Matrix = Tuple[Tuple[int, int], Tuple[int, int]]
Vec = Tuple[int, int]
Point = Tuple[int, int]

# --------------------------------------------------------------------------------------
# Closed label set for diagnosis (order is stable and part of the public contract).
# --------------------------------------------------------------------------------------

DIAGNOSIS_LABELS: List[str] = [
    "correct",
    "reflection_instead_of_rotation",
    "rotation_instead_of_reflection",
    "wrong_rotation_angle",
    "wrong_reflection_line",
    "wrong_translation",
    "opposite_translation",
    "completely_wrong",
]

# det(M) sign -> orientation class name.
ORIENTATIONS = {1: "rotation", -1: "reflection"}

IDENTITY_MATRIX: Matrix = ((1, 0), (0, 1))

# Canonical CCW rotation matrices (about the origin).
_ROTATION_MATRICES = {
    0: ((1, 0), (0, 1)),
    90: ((0, -1), (1, 0)),
    180: ((-1, 0), (0, -1)),
    270: ((0, 1), (-1, 0)),
}
_ROTATION_MATRIX_TO_DEG = {m: d for d, m in _ROTATION_MATRICES.items()}

# Reflection matrices keyed by canonical line name.
_REFLECTION_MATRICES = {
    "x": ((1, 0), (0, -1)),
    "y": ((-1, 0), (0, 1)),
    "y=x": ((0, 1), (1, 0)),
    "y=-x": ((0, -1), (-1, 0)),
}
_REFLECTION_MATRIX_TO_LINE = {m: k for k, m in _REFLECTION_MATRICES.items()}

# Human-readable line phrasing used in schema strings.
_LINE_TEXT = {
    "x": "x axis",
    "y": "y axis",
    "y=x": "line y = x",
    "y=-x": "line y = -x",
}

# All 8 lattice isometry linear parts (identity + 3 rotations + 4 reflections) and the 7
# non-identity ones (used by is_asymmetric / recover_map).
ALL_LINEAR_MAPS: Tuple[Matrix, ...] = (
    _ROTATION_MATRICES[0],
    _ROTATION_MATRICES[90],
    _ROTATION_MATRICES[180],
    _ROTATION_MATRICES[270],
    _REFLECTION_MATRICES["x"],
    _REFLECTION_MATRICES["y"],
    _REFLECTION_MATRICES["y=x"],
    _REFLECTION_MATRICES["y=-x"],
)
_NONTRIVIAL_LINEAR_MAPS: Tuple[Matrix, ...] = ALL_LINEAR_MAPS[1:]


# --------------------------------------------------------------------------------------
# Transform dataclass
# --------------------------------------------------------------------------------------

@dataclass(frozen=True)
class Transform:
    """An integer affine map ``p -> matrix @ p + vec``.

    Equality is structural (frozen dataclass) and depends ONLY on the math
    (``matrix`` and ``vec``) — never on any text wording — so net-map comparisons
    are exact.
    """

    matrix: Matrix = IDENTITY_MATRIX
    vec: Vec = (0, 0)

    def apply(self, pts: Iterable[Sequence[int]]) -> List[Point]:
        """Apply this transform to an iterable of ``(x, y)`` points."""
        (a, b), (c, d) = self.matrix
        e, f = self.vec
        return [(a * x + b * y + e, c * x + d * y + f) for x, y in pts]

    def det(self) -> int:  # noqa: D401 - short and exact
        """Determinant of the linear part (``+1`` rotation/identity, ``-1`` reflection)."""
        return matrix_det(self.matrix)

    @property
    def orientation(self) -> str:
        return ORIENTATIONS[self.det()]

    # -- matrix helpers exposed on the Transform contract -------------------------------

    @staticmethod
    def mat_mul(a: Matrix, b: Matrix) -> Matrix:
        return mat_mul(a, b)

    @staticmethod
    def mat_vec(m: Matrix, v: Sequence[int]) -> Vec:
        return mat_vec(m, v)

    @staticmethod
    def det_of(m: Matrix) -> int:
        return matrix_det(m)


# --------------------------------------------------------------------------------------
# Standalone matrix helpers (also exposed as Transform.mat_mul / mat_vec / det_of)
# --------------------------------------------------------------------------------------

def mat_mul(a: Matrix, b: Matrix) -> Matrix:
    """2x2 integer matrix product ``a @ b`` (row-major)."""
    (a00, a01), (a10, a11) = a
    (b00, b01), (b10, b11) = b
    return (
        (a00 * b00 + a01 * b10, a00 * b01 + a01 * b11),
        (a10 * b00 + a11 * b10, a10 * b01 + a11 * b11),
    )


def mat_vec(m: Matrix, v: Sequence[int]) -> Vec:
    """Apply a 2x2 matrix to a length-2 vector."""
    (a, b), (c, d) = m
    x, y = v
    return (a * x + b * y, c * x + d * y)


def matrix_det(m: Matrix) -> int:
    (a, b), (c, d) = m
    return a * d - b * c


# Convenience alias so callers can write ``transform_core.det(M)``.
det = matrix_det


# --------------------------------------------------------------------------------------
# Factories
# --------------------------------------------------------------------------------------

def identity() -> Transform:
    return Transform(IDENTITY_MATRIX, (0, 0))


def _normalize_direction(direction: str) -> str:
    d = direction.strip().lower()
    if d in ("ccw", "counterclockwise", "counter-clockwise", "anticlockwise"):
        return "ccw"
    if d in ("cw", "clockwise"):
        return "cw"
    raise ValueError(f"unknown rotation direction: {direction!r}")


def rotate(degrees: int, direction: str = "ccw") -> Transform:
    """Rotation about the origin by ``degrees`` (90/180/270) clockwise or ccw."""
    deg = int(degrees) % 360
    if deg not in (0, 90, 180, 270):
        raise ValueError(f"rotation degrees must be a multiple of 90, got {degrees!r}")
    if _normalize_direction(direction) == "cw":
        deg = (360 - deg) % 360
    return Transform(_ROTATION_MATRICES[deg], (0, 0))


def _normalize_line(line: str) -> str:
    s = line.strip().lower().replace(" ", "")
    aliases = {
        "x": "x", "xaxis": "x", "x-axis": "x", "thex-axis": "x", "thexaxis": "x",
        "y": "y", "yaxis": "y", "y-axis": "y", "they-axis": "y", "theyaxis": "y",
        "y=x": "y=x", "liney=x": "y=x", "theliney=x": "y=x",
        "y=-x": "y=-x", "liney=-x": "y=-x", "theliney=-x": "y=-x",
    }
    if s in aliases:
        return aliases[s]
    raise ValueError(f"unknown reflection line: {line!r}")


def reflect(line: str) -> Transform:
    """Reflection across one of: ``x``, ``y``, ``y=x``, ``y=-x`` (aliases accepted)."""
    return Transform(_REFLECTION_MATRICES[_normalize_line(line)], (0, 0))


def translate(dx: int, dy: int) -> Transform:
    return Transform(IDENTITY_MATRIX, (int(dx), int(dy)))


TransformLike = Union[Transform, str]


def as_transform(t: TransformLike) -> Transform:
    """Coerce a ``Transform`` or a schema string into a ``Transform``."""
    if isinstance(t, Transform):
        return t
    if isinstance(t, str):
        return parse_transform(t)
    raise TypeError(f"expected Transform or str, got {type(t).__name__}")


def compose(seq: Sequence[TransformLike]) -> Transform:
    """Compose a sequence of transforms where ``seq[0]`` is applied FIRST.

    Accepts ``Transform`` objects or schema strings (strings are parsed).
    An empty sequence composes to the identity.
    """
    result = identity()
    for item in seq:
        t = as_transform(item)
        m = mat_mul(t.matrix, result.matrix)
        v_lin = mat_vec(t.matrix, result.vec)
        v = (v_lin[0] + t.vec[0], v_lin[1] + t.vec[1])
        result = Transform(m, v)
    return result


# --------------------------------------------------------------------------------------
# Text <-> Transform
# --------------------------------------------------------------------------------------

def describe_transform(t: TransformLike, rotation_style: str = "ccw") -> str:
    """Render a single-step transform as its canonical schema string.

    ``rotation_style`` controls rotation wording only: ``"ccw"`` (default) or ``"cw"``
    (the equivalent complementary clockwise wording, e.g. ``rotate 270 degrees
    clockwise`` for a 90-degree ccw rotation). Non-rotations ignore it.
    """
    t = as_transform(t)
    if t.matrix == IDENTITY_MATRIX:
        dx, dy = t.vec
        if dx == 0 and dy == 0:
            return "identity"
        if dy == 0:
            return f"translate {abs(dx)} {'right' if dx > 0 else 'left'}"
        if dx == 0:
            return f"translate {abs(dy)} {'up' if dy > 0 else 'down'}"
        return f"translate by ({dx}, {dy})"
    if t.vec == (0, 0) and t.matrix in _ROTATION_MATRIX_TO_DEG:
        deg = _ROTATION_MATRIX_TO_DEG[t.matrix]
        style = _normalize_direction(rotation_style)
        if style == "cw":
            cw = (360 - deg) % 360
            return f"rotate {cw} degrees clockwise"
        return f"rotate {deg} degrees counterclockwise"
    if t.vec == (0, 0) and t.matrix in _REFLECTION_MATRIX_TO_LINE:
        line = _REFLECTION_MATRIX_TO_LINE[t.matrix]
        return f"reflect across {_LINE_TEXT[line]}"
    raise ValueError(f"cannot describe non-primitive transform: {t!r}")


_ROT_RE = re.compile(r"rotate\s+(\d+)\s*degrees?\s+(counterclockwise|clockwise|ccw|cw)")
_REFL_RE = re.compile(r"reflect\s+across\s+(.*)")
_TRANS_XY_RE = re.compile(r"translate\s+by\s*\(\s*(-?\d+)\s*,\s*(-?\d+)\s*\)")
_TRANS_DIR_RE = re.compile(r"translate\s+(\d+)\s+(left|right|up|down)")

_DIR_TO_VEC = {
    "left": (-1, 0),
    "right": (1, 0),
    "up": (0, 1),
    "down": (0, -1),
}


def parse_transform(text: str) -> Transform:
    """Parse a schema string back into a ``Transform`` (inverse of describe_transform)."""
    s = " ".join(text.strip().lower().split())
    m = _ROT_RE.search(s)
    if m:
        return rotate(int(m.group(1)), m.group(2))
    m = _TRANS_XY_RE.search(s)
    if m:
        return translate(int(m.group(1)), int(m.group(2)))
    m = _TRANS_DIR_RE.search(s)
    if m:
        n = int(m.group(1))
        ux, uy = _DIR_TO_VEC[m.group(2)]
        return translate(ux * n, uy * n)
    m = _REFL_RE.search(s)
    if m:
        return reflect(m.group(1))
    if s == "identity":
        return identity()
    raise ValueError(f"cannot parse transform text: {text!r}")


def describe_seq(seq: Sequence[TransformLike]) -> List[str]:
    return [describe_transform(t) for t in seq]


# --------------------------------------------------------------------------------------
# Point helpers
# --------------------------------------------------------------------------------------

def as_points(pts: Iterable[Sequence[int]]) -> List[Point]:
    """Normalize any list/tuple of coordinate pairs into a list of int tuples."""
    return [(int(x), int(y)) for x, y in pts]


# --------------------------------------------------------------------------------------
# Grading
# --------------------------------------------------------------------------------------

def grade(
    original: Iterable[Sequence[int]],
    image: Iterable[Sequence[int]],
    candidate_seq: Sequence[TransformLike],
) -> bool:
    """Return True iff applying ``candidate_seq`` (seq[0] first) to ``original`` yields ``image``."""
    produced = compose(candidate_seq).apply(as_points(original))
    return produced == as_points(image)


# --------------------------------------------------------------------------------------
# Diagnosis
# --------------------------------------------------------------------------------------

def diagnose(
    original: Iterable[Sequence[int]],
    correct_seq: Sequence[TransformLike],
    student_seq: Sequence[TransformLike],
) -> str:
    """Classify a student attempt against the correct answer.

    The classification is a total, deterministic function of the two NET affine maps
    (``original`` is accepted for API symmetry but not required — the net maps carry
    all the information). Let ``C = compose(correct_seq)``, ``S = compose(student_seq)``
    with net linear parts ``Mc, Ms`` and net translations ``tc, ts``:

    1. ``Mc == Ms and tc == ts``                      -> ``correct``
    2. ``Mc == Ms`` (linear equal, translation differs):
         * ``tc != 0 and ts == -tc``                  -> ``opposite_translation``
         * otherwise                                  -> ``wrong_translation``
    3. linear parts differ, orientations differ (``det`` sign differs):
         * translations equal, correct rotation/student reflection -> ``reflection_instead_of_rotation``
         * translations equal, correct reflection/student rotation -> ``rotation_instead_of_reflection``
         * translations also differ                   -> ``completely_wrong``
    4. linear parts differ, orientations equal:
         * both rotations, translations equal         -> ``wrong_rotation_angle``
         * both reflections, translations equal       -> ``wrong_reflection_line``
         * translations also differ                   -> ``completely_wrong``

    The result is always a member of ``DIAGNOSIS_LABELS``.
    """
    c = compose(correct_seq)
    s = compose(student_seq)
    mc, tc = c.matrix, c.vec
    ms, ts = s.matrix, s.vec
    lin_same = mc == ms
    tr_same = tc == ts

    if lin_same and tr_same:
        return "correct"

    if lin_same:  # translation differs
        if tc != (0, 0) and ts == (-tc[0], -tc[1]):
            return "opposite_translation"
        return "wrong_translation"

    # Linear parts differ from here on.
    dc, ds = matrix_det(mc), matrix_det(ms)
    if dc != ds:  # orientation confusion
        if not tr_same:
            return "completely_wrong"
        if dc == 1 and ds == -1:
            return "reflection_instead_of_rotation"
        return "rotation_instead_of_reflection"

    # Same orientation, different linear part.
    if not tr_same:
        return "completely_wrong"
    if dc == 1:
        return "wrong_rotation_angle"
    return "wrong_reflection_line"


# --------------------------------------------------------------------------------------
# Symmetry / identifiability
# --------------------------------------------------------------------------------------

def _canonical_pointset(pts: Sequence[Sequence[int]]) -> Tuple[Point, ...]:
    """Translate a point set so its min corner is at the origin, then sort — a
    translation-invariant canonical form of the (unordered) point set."""
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    mnx, mny = min(xs), min(ys)
    return tuple(sorted((int(x) - mnx, int(y) - mny) for x, y in pts))


def is_asymmetric(pts: Sequence[Sequence[int]]) -> bool:
    """True iff ``pts`` has trivial symmetry under the 8 lattice isometries modulo
    translation. An asymmetric shape guarantees that the net map taking it to any
    image is uniquely recoverable (see ``recover_map``)."""
    base = _canonical_pointset(pts)
    pts_t = as_points(pts)
    for m in _NONTRIVIAL_LINEAR_MAPS:
        mapped = [mat_vec(m, p) for p in pts_t]
        if _canonical_pointset(mapped) == base:
            return False
    return True


def recover_map(
    original: Iterable[Sequence[int]],
    image: Iterable[Sequence[int]],
) -> Optional[Transform]:
    """Recover the unique net ``Transform`` with ``T.apply(original) == image``.

    Searches the 8 lattice isometry linear parts; for each, the translation is fixed
    by the first vertex and then verified against every vertex. Returns the matching
    ``Transform`` or ``None`` if no lattice isometry maps ``original`` onto ``image``.
    For an asymmetric ``original`` the answer (if any) is unique.
    """
    orig = as_points(original)
    img = as_points(image)
    if not orig or len(orig) != len(img):
        return None
    for m in ALL_LINEAR_MAPS:
        mx, my = mat_vec(m, orig[0])
        t = (img[0][0] - mx, img[0][1] - my)
        ok = True
        for p, q in zip(orig, img):
            px, py = mat_vec(m, p)
            if (px + t[0], py + t[1]) != q:
                ok = False
                break
        if ok:
            return Transform(m, t)
    return None


In [ ]:
%%writefile slm_eval/hints.py
"""hints — deterministic, templated tutor hints for diagnosis records.

Every hint is derived from the record's OWN transforms via
``transform_core.describe_transform`` (never hand-typed axis / angle / vector
literals), so the wording always names the correct/student axis, angle, or
translation for that record. There is no second transform implementation here —
this module only parses the stored schema strings back into
``transform_core.Transform`` objects and re-describes them canonically.

Two public functions:

* :func:`hint_for` — build the deterministic hint string for a record.
* :func:`expected_hint_tokens` — the deterministically-expected substring
  token(s) the hint MUST contain for a label. Used by
  ``dataset._assert_record`` to enforce hint correctness at write time.

Both take ``(label, rec)`` where ``rec`` is any mapping exposing the two schema
key lists ``"correct_transform"`` and ``"student_transform"`` (a full record or
the partial built in :mod:`dataset`). They agree by construction: ``hint_for``
embeds exactly the strings ``expected_hint_tokens`` recomputes.
"""

from __future__ import annotations

from typing import List, Mapping, Sequence

from . import transform_core as tc


# --------------------------------------------------------------------------------------
# Small helpers (all wording flows through transform_core.describe_transform)
# --------------------------------------------------------------------------------------

def _parse(seq: Sequence) -> List[tc.Transform]:
    return [tc.as_transform(s) for s in seq]


def _desc(t: tc.Transform) -> str:
    """Canonical single-step description (ccw rotations) via the single source of truth."""
    return tc.describe_transform(t)


def _vec_str(vec) -> str:
    """Render a net translation vector as a literal ``(dx, dy)`` token."""
    return f"({vec[0]}, {vec[1]})"


def _correct_student(rec: Mapping):
    return _parse(rec["correct_transform"]), _parse(rec["student_transform"])


def _diff_indices(correct: Sequence[tc.Transform], student: Sequence[tc.Transform]) -> List[int]:
    """Positions where the correct and student STEPS differ (by exact math, not wording)."""
    n = min(len(correct), len(student))
    return [i for i in range(n) if correct[i] != student[i]]


def _single_diff(correct, student) -> int:
    """Index of the one step a single-step error injector mutated (fails loudly otherwise)."""
    idxs = _diff_indices(correct, student)
    if len(idxs) != 1:
        raise ValueError(f"expected exactly one differing step, got {idxs}")
    return idxs[0]


# --------------------------------------------------------------------------------------
# Per-label token extraction (the deterministically-expected substrings)
# --------------------------------------------------------------------------------------

def expected_hint_tokens(label: str, rec: Mapping) -> List[str]:
    """Return the substring token(s) a correct hint for ``label`` MUST contain.

    Every token is recomputed from the record's transforms through
    ``transform_core`` — the correct/student axis, angle, or translation — so
    asserting each is a substring of the hint verifies the hint truly names the
    right thing for this record.
    """
    correct, student = _correct_student(rec)

    if label == "correct":
        return [_desc(correct[0]), _desc(correct[1])]

    if label == "completely_wrong":
        c = tc.compose(correct)
        s = tc.compose(student)
        return [
            _desc(tc.Transform(c.matrix, (0, 0))),
            _desc(tc.Transform(s.matrix, (0, 0))),
            _vec_str(c.vec),
            _vec_str(s.vec),
        ]

    # All remaining labels are single-step mutations: name the correct step and
    # the student's substituted step.
    i = _single_diff(correct, student)
    return [_desc(correct[i]), _desc(student[i])]


# --------------------------------------------------------------------------------------
# Hint construction
# --------------------------------------------------------------------------------------

def hint_for(label: str, rec: Mapping) -> str:
    """Build the deterministic, templated hint for ``rec`` given its ``label``.

    The returned string references the concrete step(s) involved, worded via
    ``transform_core.describe_transform`` so the axis / angle / translation named
    always matches the record's actual transforms.
    """
    if label not in tc.DIAGNOSIS_LABELS:
        raise ValueError(f"unknown label: {label!r}")

    correct, student = _correct_student(rec)

    if label == "correct":
        s0, s1 = _desc(correct[0]), _desc(correct[1])
        return (
            f"Correct. Both steps are right: first {s0}, then {s1}. "
            f"You applied each move in the right order."
        )

    if label == "reflection_instead_of_rotation":
        i = _single_diff(correct, student)
        return (
            f"You reflected where a rotation was required. The step should have "
            f"been {_desc(correct[i])}, not {_desc(student[i])}."
        )

    if label == "rotation_instead_of_reflection":
        i = _single_diff(correct, student)
        return (
            f"You rotated where a reflection was required. The step should have "
            f"been {_desc(correct[i])}, not {_desc(student[i])}."
        )

    if label == "wrong_rotation_angle":
        i = _single_diff(correct, student)
        return (
            f"Check the angle: the task used {_desc(correct[i])}, but you used "
            f"{_desc(student[i])}."
        )

    if label == "wrong_reflection_line":
        i = _single_diff(correct, student)
        return (
            f"Check the line of reflection: the task used {_desc(correct[i])}, but "
            f"you used {_desc(student[i])}."
        )

    if label == "wrong_translation":
        i = _single_diff(correct, student)
        return (
            f"Check the translation: it should be {_desc(correct[i])}, not "
            f"{_desc(student[i])}."
        )

    if label == "opposite_translation":
        i = _single_diff(correct, student)
        return (
            f"You translated in the opposite direction: it should be "
            f"{_desc(correct[i])}, not {_desc(student[i])}."
        )

    # completely_wrong: name the correct vs student NET map (orientation/linear
    # part AND translation both differ).
    c = tc.compose(correct)
    s = tc.compose(student)
    c_lin = _desc(tc.Transform(c.matrix, (0, 0)))
    s_lin = _desc(tc.Transform(s.matrix, (0, 0)))
    return (
        f"Your whole answer is off: the correct net map is a {c.orientation} "
        f"({c_lin}) with translation {_vec_str(c.vec)}, but yours is a "
        f"{s.orientation} ({s_lin}) with translation {_vec_str(s.vec)}. Both the "
        f"transformation and the translation are wrong."
    )


In [ ]:
%%writefile slm_eval/eval.py
"""eval — programmatic, judge-free scoring of model diagnoses against the oracle.

The vision fine-tune emits a single JSON object ``{"label", "correct_transform", "hint"}``
per record (see :mod:`chat_format`). Because every record carries a ground-truth answer
verified by :mod:`transform_core` (and the exact hint substrings via
:func:`hints.expected_hint_tokens`), grading is fully deterministic — no LLM-as-judge, no
hand-labeling. Each record decomposes into four yes/no checks:

    parse_ok      valid JSON with a known label
    label_ok      predicted label == oracle label
    transform_ok  predicted correct_transform composes to the same net map as the oracle
    hint_ok       hint names the right error (expected tokens present) AND leaks no
                  coordinates it wasn't sanctioned to state

We report the fraction passing each check plus balanced accuracy and an 8x8 confusion
matrix. This module is pure Python (no torch / PIL) so it unit-tests locally and imports
cleanly inside the Colab notebook.

Split discipline lives in the caller: run this over the frozen ``test`` + ``ood`` splits
for the headline base-vs-tuned number, and over ``val`` for iteration. This module does not
know or care which split it is handed.
"""

from __future__ import annotations

import json
import re
from collections import Counter
from typing import Dict, List, Optional, Sequence

from . import hints
from . import transform_core as tc

# Per-record row schema saved to JSONL for error analysis.
RECORD_FIELDS = (
    "id",
    "split",
    "true_label",
    "pred_label",
    "parse_ok",
    "label_ok",
    "transform_ok",
    "hint_ok",
    "raw_model_output",
    "failure_reason",
)

_PARSE_FAIL = "PARSE_FAIL"
# A coordinate pair literal, e.g. "(3, -4)" — used only for residual-leak detection.
_COORD_RE = re.compile(r"\(-?\d+,-?\d+\)")


# --------------------------------------------------------------------------------------
# Prediction parsing
# --------------------------------------------------------------------------------------

def parse_pred(text: str) -> Optional[dict]:
    """Extract the single JSON object from a model output, or ``None``.

    Tolerates surrounding prose and ```` ``` ```` code fences: tries a direct
    ``json.loads`` first, then falls back to the first brace-balanced ``{...}`` span.
    """
    if not isinstance(text, str):
        return None
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\n?", "", s)
        s = re.sub(r"\n?```$", "", s).strip()
    try:
        obj = json.loads(s)
    except (ValueError, TypeError):
        obj = _first_json_object(s)
    return obj if isinstance(obj, dict) else None


def _first_json_object(s: str) -> Optional[dict]:
    start = s.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(s)):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(s[start : i + 1])
                except ValueError:
                    return None
    return None


# --------------------------------------------------------------------------------------
# Field checks
# --------------------------------------------------------------------------------------

def _norm(text: str) -> str:
    """Lowercase + collapse whitespace — for tolerant substring matching."""
    return " ".join(str(text).lower().split())


def _transform_match(pred_transform: object, gold_transform: Sequence[str]) -> bool:
    """True iff ``pred_transform`` composes to the same net affine map as the oracle.

    Semantic (compose-and-compare via :mod:`transform_core`), so wording variants that
    mean the same motion still pass. Falls back to a normalized exact-string list match
    when a step can't be parsed.
    """
    if isinstance(pred_transform, str):
        pred_transform = [pred_transform]
    if not isinstance(pred_transform, (list, tuple)):
        return False
    try:
        return tc.compose(pred_transform) == tc.compose(gold_transform)
    except (ValueError, TypeError):
        pass
    # Unparseable step: last-resort exact (normalized) string comparison.
    if len(pred_transform) != len(gold_transform):
        return False
    return all(_norm(a) == _norm(b) for a, b in zip(pred_transform, gold_transform))


def _hint_tokens_present(pred_hint: str, tokens: Sequence[str]) -> bool:
    hint = _norm(pred_hint)
    return all(_norm(tok) in hint for tok in tokens)


def _hint_has_leak(pred_hint: str, tokens: Sequence[str]) -> bool:
    """True iff the hint states a coordinate pair it was NOT sanctioned to name.

    Sanctioned pairs (e.g. the ``(dx, dy)`` net-translation tokens in a
    ``completely_wrong`` hint) come from ``expected_hint_tokens`` and are removed before
    scanning, so only *extra* coordinates — the kind that would give the answer away —
    count as a leak.
    """
    residual = re.sub(r"\s+", "", pred_hint.lower())
    for tok in tokens:
        residual = residual.replace(re.sub(r"\s+", "", tok.lower()), "")
    return bool(_COORD_RE.search(residual))


# --------------------------------------------------------------------------------------
# Per-record scoring
# --------------------------------------------------------------------------------------

def score_record(pred_text: str, rec: dict) -> Dict[str, object]:
    """Score one model output against its oracle record. Returns a ``RECORD_FIELDS`` row.

    ``hint`` is always graded against the record's TRUE label (the hint should describe
    the real diagnosis), so a wrong-label prediction fails ``hint_ok`` too — as it should.
    """
    true_label = rec["label"]
    base = {
        "id": rec.get("id"),
        "split": rec.get("split"),
        "true_label": true_label,
        "raw_model_output": pred_text,
    }

    pred = parse_pred(pred_text)
    if pred is None or pred.get("label") not in tc.DIAGNOSIS_LABELS:
        return {
            **base,
            "pred_label": _PARSE_FAIL,
            "parse_ok": False,
            "label_ok": False,
            "transform_ok": False,
            "hint_ok": False,
            "failure_reason": "parse_fail",
        }

    pred_label = pred["label"]
    label_ok = pred_label == true_label
    transform_ok = _transform_match(pred.get("correct_transform"), rec["correct_transform"])

    tokens = hints.expected_hint_tokens(true_label, rec)
    pred_hint = pred.get("hint", "") or ""
    tokens_present = _hint_tokens_present(pred_hint, tokens)
    hint_leak = _hint_has_leak(pred_hint, tokens)
    hint_ok = tokens_present and not hint_leak

    # Primary failure, by priority (per-field booleans above carry the full detail).
    if not label_ok:
        reason = f"wrong_label:{true_label}->{pred_label}"
    elif not transform_ok:
        reason = "transform_mismatch"
    elif hint_leak:
        reason = "hint_leak"
    elif not tokens_present:
        reason = "hint_missing_token"
    else:
        reason = ""

    return {
        **base,
        "pred_label": pred_label,
        "parse_ok": True,
        "label_ok": label_ok,
        "transform_ok": transform_ok,
        "hint_ok": hint_ok,
        "failure_reason": reason,
    }


def score_all(pred_texts: Sequence[str], recs: Sequence[dict]) -> List[Dict[str, object]]:
    """Score parallel sequences of model outputs and oracle records."""
    if len(pred_texts) != len(recs):
        raise ValueError(f"length mismatch: {len(pred_texts)} preds vs {len(recs)} records")
    return [score_record(p, r) for p, r in zip(pred_texts, recs)]


# --------------------------------------------------------------------------------------
# Aggregation
# --------------------------------------------------------------------------------------

def _mean(flags: Sequence[bool]) -> float:
    return sum(1 for f in flags if f) / len(flags) if flags else 0.0


def aggregate(results: Sequence[dict], labels: Optional[Sequence[str]] = None) -> dict:
    """Roll per-record rows up into the reported metrics.

    ``balanced_accuracy`` is the mean per-label recall over only the labels actually
    present in ``results`` (so it's meaningful on the OOD split, which carries 4 of 8
    labels). ``confusion[true][pred]`` counts, with ``pred == "PARSE_FAIL"`` for outputs
    that didn't parse to a known label.
    """
    labels = list(labels or tc.DIAGNOSIS_LABELS)
    n = len(results)
    if n == 0:
        return {"n": 0}

    per_label_recall: Dict[str, Optional[float]] = {}
    for lab in labels:
        subset = [r for r in results if r["true_label"] == lab]
        per_label_recall[lab] = _mean([r["label_ok"] for r in subset]) if subset else None

    present = [v for v in per_label_recall.values() if v is not None]
    balanced_accuracy = sum(present) / len(present) if present else 0.0

    confusion: Dict[str, Counter] = {lab: Counter() for lab in labels}
    for r in results:
        confusion.setdefault(r["true_label"], Counter())[r["pred_label"]] += 1

    return {
        "n": n,
        "parse_rate": _mean([r["parse_ok"] for r in results]),
        "label_accuracy": _mean([r["label_ok"] for r in results]),
        "balanced_accuracy": balanced_accuracy,
        "transform_match_rate": _mean([r["transform_ok"] for r in results]),
        "hint_match_rate": _mean([r["hint_ok"] for r in results]),
        "per_label_recall": per_label_recall,
        "confusion": {t: dict(c) for t, c in confusion.items()},
    }


# --------------------------------------------------------------------------------------
# Reporting
# --------------------------------------------------------------------------------------

_TABLE_METRICS = (
    ("parse_rate", "parse_rate"),
    ("label_accuracy", "label_acc"),
    ("balanced_accuracy", "balanced_acc"),
    ("transform_match_rate", "transform_match"),
    ("hint_match_rate", "hint_match"),
)


def format_table(base: dict, tuned: dict) -> str:
    """A base-vs-tuned metrics table (the headline deliverable)."""
    rows = [f"{'metric':<18}{'base':>8}{'tuned':>8}{'delta':>8}"]
    rows.append("-" * 42)
    for key, label in _TABLE_METRICS:
        b, t = base.get(key, 0.0), tuned.get(key, 0.0)
        rows.append(f"{label:<18}{b:>8.3f}{t:>8.3f}{t - b:>+8.3f}")
    return "\n".join(rows)


def format_confusion(agg: dict, labels: Optional[Sequence[str]] = None) -> str:
    """Render the confusion matrix as a text grid (true rows x predicted cols).

    Columns are the label set plus ``PF`` (PARSE_FAIL). Uses short numeric codes for
    labels (legend printed above) so the grid stays readable in a notebook cell.
    """
    labels = list(labels or tc.DIAGNOSIS_LABELS)
    confusion = agg.get("confusion", {})
    cols = labels + [_PARSE_FAIL]
    codes = {lab: str(i) for i, lab in enumerate(labels)}
    codes[_PARSE_FAIL] = "PF"

    legend = "  ".join(f"{codes[lab]}={lab}" for lab in labels)
    header = "true\\pred".ljust(12) + "".join(codes[c].rjust(5) for c in cols)
    lines = [legend, "", header]
    for lab in labels:
        row = confusion.get(lab, {})
        cells = "".join(str(row.get(c, 0)).rjust(5) for c in cols)
        lines.append(codes[lab].ljust(12) + cells)
    return "\n".join(lines)


def save_results(agg: dict, agg_path: str, records: Sequence[dict], records_path: str) -> None:
    """Write the aggregate metrics (JSON) and the per-record rows (JSONL).

    The per-record file is what turns "the model improved" into "*this* behavior improved,
    and it still fails *here*" — one line per record with the ``RECORD_FIELDS`` schema.
    """
    with open(agg_path, "w") as f:
        json.dump(agg, f, indent=2)
    with open(records_path, "w") as f:
        for r in records:
            row = {k: r.get(k) for k in RECORD_FIELDS}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


In [ ]:
from slm_eval import eval as ev, transform_core as tc
LABELS = tc.DIAGNOSIS_LABELS      # single source of truth for the 8-label vocab
print("eval harness ready; labels:", LABELS)

In [ ]:
# --- Get your dataset onto Colab (the only thing you upload besides this notebook) --------
# Drag `transform_diagnosis_data.zip` into the Files pane (folder icon, left), then run this
# cell: it finds and unzips it. (Alternatively mount Drive and set DATA_DIR to the folder.)
import os, glob, subprocess

DATA_DIR = "transform_diagnosis_data"
if not os.path.isdir(DATA_DIR):
    zips = glob.glob("*.zip") + glob.glob("/content/*.zip")
    if zips:
        print("unzipping", zips[0])
        subprocess.run(["unzip", "-q", zips[0], "-d", "/content"], check=False)
        if os.path.isdir("/content/transform_diagnosis_data"):
            DATA_DIR = "/content/transform_diagnosis_data"

assert os.path.isdir(DATA_DIR), (
    "No dataset found. Upload transform_diagnosis_data.zip into the Files pane, then re-run "
    "this cell."
)
print("data dir:", DATA_DIR)
print("splits:", sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".jsonl")))

In [ ]:
from unsloth import FastVisionModel
import torch

MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(model)  # enable Unsloth's faster inference path
print("loaded:", MODEL_NAME)

In [ ]:
# --- Load conversations LAZILY (keep image PATHS; do NOT decode 26k images into RAM) ------
# Each 446x436 render is ~0.58 MB decoded, so decoding all 26k up front is ~15 GB and crashes
# a free-Colab kernel (~12 GB RAM). So we keep the image PATH in each message and decode on
# demand: a small TRAIN subset (next cell) and one image at a time during eval.
import json, copy
from PIL import Image

def load_split(split):
    """Chat rows with image PATHS left as strings (cheap — no pixels held in RAM)."""
    rows = []
    with open(os.path.join(DATA_DIR, f"{split}_chat.jsonl")) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_rows = load_split("train")
val_rows   = load_split("val")
test_rows  = load_split("test")
ood_rows   = load_split("ood")
print({s: len(r) for s, r in
       [("train", train_rows), ("val", val_rows), ("test", test_rows), ("ood", ood_rows)]})

def decode_messages(messages):
    """Deep-copy a messages list and swap each image PATH for a decoded PIL image."""
    msgs = copy.deepcopy(messages)
    for msg in msgs:
        for part in msg["content"]:
            if part.get("type") == "image" and isinstance(part.get("image"), str):
                part["image"] = Image.open(
                    os.path.join(DATA_DIR, part["image"])
                ).convert("RGB")
    return msgs

def ground_truth(rec):
    """The record's own assistant turn is the gold target JSON."""
    return json.loads(rec["messages"][1]["content"][0]["text"])

print("example gold target:", ground_truth(train_rows[0]))

In [ ]:
# --- Build the FULL train set (all 19,200 images) -----------------------------------------
# Decoding all of them holds ~11 GB in RAM — fine on this 64 GB node (would crash a 12 GB
# free-Colab; there you'd set N_TRAIN to a few thousand). This decode takes ~1-2 minutes.
N_TRAIN = len(train_rows)          # full training set
train_dataset = [{"messages": decode_messages(r["messages"])} for r in train_rows[:N_TRAIN]]
print("train examples (decoded):", len(train_dataset), "of", len(train_rows))

# Format-fit check: render one conversation through the chat template. You should see the Qwen
# vision placeholders (<|vision_start|> ... <|vision_end|>) and the JSON target as the answer.
proof = tokenizer.apply_chat_template(train_dataset[0]["messages"], tokenize=False)
print(proof[:1200])

In [ ]:
# --- Inference + programmatic scoring via the eval harness --------------------------------
# run_model: generate the raw diagnosis string for one record's user turn (GPU).
# Scoring is done by the harness (slm_eval/eval.py): each output -> yes/no on
# parse_ok / label_ok / transform_ok / hint_ok, plus balanced accuracy and an 8x8 confusion.
# We grade against the FULL raw record (test.jsonl / ood.jsonl), because hint-token checking
# needs `student_transform`, which the *_chat.jsonl rows do not carry.
# Images are decoded ONE AT A TIME here (lazy) so eval adds ~no persistent RAM.

EVAL_N = None   # None = score the FULL frozen split (2400 test + 2000 ood). On the L40S each
                # short JSON generation is fast, so this is fine. If the 6 h session looks
                # tight, set e.g. EVAL_N = 1000 (use the SAME value for base and tuned).

def run_model(user_message, image, max_new_tokens=256):
    """Generate the model's raw diagnosis string for one record's user turn."""
    input_text = tokenizer.apply_chat_template([user_message], add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def load_raw(split):
    """Full oracle records for a split, keyed by id (carry correct_transform,
    student_transform, hint — everything score_record needs). JSON only, cheap."""
    by_id = {}
    with open(os.path.join(DATA_DIR, f"{split}.jsonl")) as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                by_id[r["id"]] = r
    return by_id

def run_eval(tag, chat_rows, raw_by_id, n=EVAL_N):
    """Score the model over chat_rows against the oracle, save results, return the aggregate.

    Writes results_<tag>.json (metrics) and records_<tag>.jsonl (one scored row per record,
    with raw_model_output + failure_reason — the file you read for error analysis).
    """
    rows = chat_rows if n is None else chat_rows[:n]
    scored = []
    for row in rows:
        user_msg = decode_messages(row["messages"])[0]   # decode THIS row's image only
        image = next(p["image"] for p in user_msg["content"] if p.get("type") == "image")
        text = run_model(user_msg, image)
        scored.append(ev.score_record(text, raw_by_id[row["id"]]))
    agg = ev.aggregate(scored)
    ev.save_results(agg, f"results_{tag}.json", scored, f"records_{tag}.jsonl")
    print(f"[{tag}] n={agg['n']} label_acc={agg['label_accuracy']:.3f} "
          f"balanced_acc={agg['balanced_accuracy']:.3f} parse={agg['parse_rate']:.3f} "
          f"transform={agg['transform_match_rate']:.3f} hint={agg['hint_match_rate']:.3f}")
    return agg

# Oracle records for the frozen evaluation splits (+ val, used only for Day-4 iteration).
raw_test = load_raw("test")
raw_ood  = load_raw("ood")
raw_val  = load_raw("val")
print("raw records:", {"test": len(raw_test), "ood": len(raw_ood), "val": len(raw_val)})

In [ ]:
# --- Baseline (before fine-tune) on the FROZEN test + ood splits --------------------------
# The number the fine-tune must beat. test/ood are frozen: scored here, and exactly once more
# after training. ALL iteration/failure-reading happens on `val` — never tune against test.
FastVisionModel.for_inference(model)
base_test = run_eval("base_test", test_rows, raw_test)
base_ood  = run_eval("base_ood",  ood_rows,  raw_ood)

In [ ]:
# --- QLoRA vision fine-tune (full epoch over all 19,200) ----------------------------------
from unsloth import FastVisionModel, is_bf16_supported
try:
    from unsloth import UnslothVisionDataCollator
except Exception:
    from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    random_state=3407, use_rslora=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,               # full pass over all 19,200 (~2,400 steps on the L40S)
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        # vision-specific: keep raw messages and let the collator build the batches
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=1,
        max_seq_length=2048,
    ),
)
trainer.train()

In [ ]:
# --- Save the LoRA adapters NOW (so a Colab disconnect can't cost you the training run) ----
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")
print("saved to lora_adapters/  (download it, or copy to Drive:)")
print("  from google.colab import drive; drive.mount('/content/drive')")
print("  !cp -r lora_adapters /content/drive/MyDrive/")

In [ ]:
# --- After fine-tune: re-score the SAME frozen test + ood splits --------------------------
# OOD = the two rotation-reflection compositions never seen in training; it probes
# compositional generalization. Its slice is unbalanced, so read balanced_acc there.
FastVisionModel.for_inference(model)
tuned_test = run_eval("tuned_test", test_rows, raw_test)
tuned_ood  = run_eval("tuned_ood",  ood_rows,  raw_ood)

In [ ]:
# --- Deliverable: base-vs-tuned tables + confusion + failure breakdown --------------------
import collections

print("== IN-DISTRIBUTION TEST (base vs tuned) ==")
print(ev.format_table(base_test, tuned_test))
print("\n== OOD / held-out compositions (base vs tuned) ==")
print(ev.format_table(base_ood, tuned_ood))

print("\n== Tuned TEST confusion (true rows x predicted cols; PF = parse fail) ==")
print(ev.format_confusion(tuned_test))
print("\n== Tuned OOD confusion ==")
print(ev.format_confusion(tuned_ood))

print("\n== Per-label recall (tuned test) ==")
for lab, rec in tuned_test["per_label_recall"].items():
    print(f"  {lab:34s} {'--' if rec is None else f'{rec:.3f}'}")

# Error analysis: the worst failure modes, read straight from the saved per-record rows.
fails = collections.Counter()
with open("records_tuned_test.jsonl") as f:
    for line in f:
        r = json.loads(line)
        if r["failure_reason"]:
            kind = r["failure_reason"].split(":")[0]      # collapse wrong_label:X->Y
            fails[(r["true_label"], kind)] += 1
print("\n== Top tuned-test failure modes (true_label, kind) ==")
for (lab, kind), c in fails.most_common(12):
    print(f"  {c:4d}  {lab:34s} {kind}")
print("\nSaved: results_/records_ {base,tuned}_{test,ood}. Do error analysis + data fixes on"
      " `val` (raw_val), retrain, then re-run the two eval cells above for the final numbers.")

## Reading the results

Four **binary checks** per output, reported as % passing: `parse_rate`, `label_acc`,
**`balanced_acc`** (mean per-label recall — the honest headline on the unbalanced OOD slice),
`transform_match`, `hint_match`. Each run saves `results_<tag>.json` + `records_<tag>.jsonl`
(every prediction + `failure_reason`) for `tag ∈ {base,tuned}_{test,ood}`.

**This notebook is configured for a FULL run** (all 19,200 training images, 1 epoch, full
frozen test+ood eval) — sized for a 44 GB GPU + 64 GB RAM node like MIT ORCD. Just Run All.

**Split discipline:** `test`/`ood` are frozen (scored twice: baseline + final). Do all
iteration and failure-reading on `val` (`raw_val` is loaded for it).

## Notes
- **Adapters are saved right after training** (`lora_adapters/`) — before the post-tune eval —
  so even if the session ends mid-eval, the trained model is safe. Copy it somewhere durable:
  `!cp -r lora_adapters ~/` (it's already in your home dir here) or push to the Hugging Face Hub.
- **If the 6 h session looks tight:** set `EVAL_N = 1000` in the defs cell (same value for base
  and tuned). Training the full epoch is the priority; eval can run on a large fixed sample.
- **On a small (12 GB) machine instead:** set `N_TRAIN` to ~4000 in the train-set cell so the
  image decode fits in RAM.